# tensor张量

In [3]:
from torchvision import transforms
import torch
from PIL import Image
img_path = "/Users/haoye/anaconda_projects/basic_learning/pytorch/CIFAR10模型训练/imgs/cat_buding.png"
img = Image.open(img_path)#PIL打开库方法就是Image.open()
# 你的图片经过 ToTensor 后
transform = transforms.ToTensor()
img_tensor = transform(img)

# 查看参数
print(f"形状: {img_tensor.shape}")      # torch.Size([3, 32, 32])
print(f"数据类型: {img_tensor.dtype}")  # torch.float32
print(f"设备: {img_tensor.device}")     # cpu
print(f"值范围: {img_tensor.min():.2f} ~ {img_tensor.max():.2f}")  # 0.00 ~ 1.00

形状: torch.Size([3, 1920, 1440])
数据类型: torch.float32
设备: cpu
值范围: 0.00 ~ 1.00


# transform
**指的是transform.py，相当于是一个工具箱,能够把数据类型转换成tensor（totensor）还有resize等不同工具（class）拿特定格式的图片经过这个工具就会输出想要的结果** 

### ToTensor（）的用法
**把 PIL 图片或 NumPy 数组转换成 PyTorch 张量，并自动归一化（-1·1）**  
`train_data = torchvision.datasets.CIFAR10(root="data", train=True, transform=torchvision.transforms.ToTensor(), download=True)`  
**返回的类型是tensor**
* tool = transform.Totensor() 创建一个转换工具不用每次都调用ToTensor类使得代码更简洁
* tensor = tool(img) 调用tool对象里面的__call__方法 将传进来的PIL或numpy转为tensor

### 其他一些类的用法

In [3]:
from torchvision import transforms 
from torch.utils.tensorboard import SummaryWriter
from PIL import Image


# 配合tensorboard可视化界面
writer = SummaryWriter("logs")

# 打开图片先
img = Image.open("/Users/haoye/anaconda_projects/basic_learning/pytorch/CIFAR10模型训练/imgs/cat_buding.png")

# ToTensor
trans_tool = transforms.ToTensor() #把类实例化，让代码更简洁不用每次都要调用。性能差不多
img_tensor = trans_tool(img)

# Normalize
print(img_tensor[0][0][0])
trans_norm = transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
## 第一个列表 [0.5, 0.5, 0.5] 是均值 (mean)第二个列表 [0.5, 0.5, 0.5] 是标准差 (std)
## 含义：对 RGB 三个通道分别减去 0.5，再除以 0.5，值域会变成【-1到1】梯度更稳定加速模型收敛
## 执行计算：output = (input - mean) / std
img_norm = trans_norm(img_tensor)
print(img_norm[0][0][0])

# Resize
## resize并不是真的改了图片的格式，而是为了方便训练，内存上张量被临时改变了尺寸
print(img_norm.shape)
trans_resize = transforms.Resize((512,512)) #resize的参数是目标参数而不是图像数据
img_resize = trans_resize(img_norm) #可以接受参数是PIL型和tensor型
print(type(img_resize))
print(img_resize.shape)# PIL型是.size,totensor型是.shape。

# compose
## compose()里面需要的是由transform类型组成的参数列表使得各类按顺序执行。代码更简洁维护更方便
## 可以把compose看成是一个容器，里面装的是transforms文件下的各种类(实例对象)
trans_compose = transforms.Compose([trans_resize,trans_tool])
img_compose = trans_compose(img)

# RandomCrop
## 按照格式来裁剪
trans_random =transforms.RandomCrop(200)
trans_compose_2 = transforms.Compose([trans_random,trans_tool])
for  i in range(10):
    img_crop =trans_compose_2(img)
    writer.add_image("randomcrop",img_crop,i)



# 配合tensorboard可视化界面
writer.add_image("random",img_crop,1)
writer.close()


tensor(0.3804)
tensor(-0.2392)
torch.Size([3, 1920, 1440])
<class 'torch.Tensor'>
torch.Size([3, 512, 512])


# torchvision中数据集的使用

In [9]:
import torchvision  #引入命名空间
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

#  第一步写预处理流程，把数据集的内容转成张量
dataset_transform = transforms.Compose([
    transforms.ToTensor()
]) 

# 第二步，把数据集传入
train_set = torchvision.datasets.CIFAR10(root = "./dataset",train = True,transform=dataset_transform, download = False)#传预处理流程参数
test_set  = torchvision.datasets.CIFAR10(root = "./dataset",train = False,transform=dataset_transform, download = False)
#print(train_set.classes) #文字标签，人看的。
#print(train_set.targets)#数字标签，计算机看的。

#第三步，既然数据集已经是tensor了所以用tensorboard可视化
writer = SummaryWriter("logs")
for i in range(10):
    # CIFAR10 数据集中，每个索引对应一个元组
    # 元组可以返回其中的元素，但只能通过索引或解包，不能通过属性名（如 .targets）。获取某个元素必须要拆包或者索引
    #   label_9 = train_set.targets[9] #对的，返回的是数据集的targets的第九个
    #   label_8 = train_set[9].targets # 错的，返回的是数据集第九个元组的targets，而元组没有targets
    #    image, label = train_set[9] 既然返回的是元组的话那就得解包把label解出来（提前知道有这个属性）
    img,target = test_set[i]
    writer.add_image("test_set",img,i) # 别名，tensor数据类型，步长
writer.close()



# dataloader  
**dataloader按照返回的格式比如返回的是img，target的话，会把数据集所有数据的img打包成imgs，target打包成targets**

In [ ]:
import torchvision 
import torchvision.transforms as transform
from torch.utils.tensorboard import SummaryWriter
from  torch.utils.data import DataLoader 

#准备的测试集
dataset_transform = transform.Compose([transform.ToTensor()])
test_set = torchvision.datasets.CIFAR10(root = "./dataset",train = False,transform=dataset_transform, download = True)

#开始dataloader
test_loader =DataLoader(dataset=test_set,batch_size=64,shuffle=True,num_workers=0,drop_last=True)
#batch_size是指每次从数据集中取多少个数据
#drop_last如果设置的是true，那么最后一个批次的数据如果不齐将会被舍弃
#shuffle 参数控制的是每个 epoch（遍历一次全部数据）之前，是否将数据顺序打乱(True)。它直接影响模型训练的效果和稳定性。

writer = SummaryWriter("logs")
for epoch in range(2): #循环两次
    step = 0
    for data in test_loader: #data存储的是每次迭代返回的一个批次的数据
        imgs,targets =data
        writer.add_images("Epoch:{}".format(epoch),imgs,step)#一批有多个图片叠加，得用add_images
        step += 1     #step++是错的python没有这种写法
writer.close()


#for data in test_loader:
    #imgs,targets = data
    #print(imgs.shape)
    #print(targets)
    ##torch.size第一维表示批次有多少元素，第二维表示通道数，第三四维表示图片是几乘几的
    ##tensor 表示的是各个图片的标签批次
